# Regression: daily alcohol intake from a liver enzyme panel

This notebook walks through the regression in the order the coursework took
it. It calls the same functions as `analysis/m04_regression.py`, and the
write-up with every table and figure is
[docs/04-regression.md](../docs/04-regression.md).

The BUPA liver disorders data (Forsyth 1990) records five blood tests and one
behavioral measurement, half-pints of alcohol reported per day, for 345 male
subjects. The seventh released column, `selector`, is a train and test split
flag that most of the published literature has read as a disease label
(McDermott and Forsyth 2016). It is dropped before anything is fitted.

In [1]:
import os
import sys
import tempfile
from pathlib import Path

# Every output directory is redirected to a temporary location before the
# pipeline modules are imported, so this notebook writes nothing into
# results/, figures/ or data/processed/. The committed tables are read from
# results/ directly where the notebook quotes them.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
_scratch = tempfile.mkdtemp()
for _name in ("ML_METHODS_RESULTS", "ML_METHODS_FIGURES", "ML_METHODS_PROCESSED"):
    os.environ[_name] = _scratch
sys.path.insert(0, str(ROOT))

import numpy
import pandas

from src import config, data, evaluate, splits

RESULTS = ROOT / "results"
pandas.set_option("display.width", 120)
pandas.set_option("display.max_columns", 20)


def recorded(prefix=""):
    """The committed metrics record, as a dictionary of strings."""
    table = pandas.read_csv(RESULTS / "metrics.csv", dtype=str)
    return {row.key: row.value for row in table.itertuples()
            if row.key.startswith(prefix)}

## Data and cleaning

`data.load_bupa` confirms the released file still carries `selector`, drops
it, and collapses the four rows that are identical in every column.

In [2]:
frame, counts = data.load_bupa()
print(pandas.Series(counts))
frame.describe().round(2)

rows_released               345
selector_distinct_values      2
rows_duplicated               4
rows_analyzed               341
drinks_zero_rows              9
dtype: int64


,mcv,alkphos,sgpt,sgot,gammagt,drinks
count,341.00,341.00,341.00,341.00,341.00,341.00
mean,90.12,69.89,30.51,24.66,38.40,3.43
std,4.45,18.43,19.59,10.12,39.44,3.34
min,65.00,23.00,4.00,5.00,5.00,0.00
25%,87.00,57.00,19.00,19.00,15.00,0.50
50%,90.00,67.00,26.00,23.00,25.00,3.00
75%,92.00,80.00,34.00,27.00,46.00,5.00
max,103.00,138.00,155.00,82.00,297.00,20.00


## Inputs and the partition

Five predictors and one target. The partition is 255 subjects for training
and 86 for testing, drawn over rows at the recorded seed, and the overlap
check compares row content so a duplicate could not sit on both sides.

In [3]:
predictors = list(config.BUPA_PREDICTORS)
target = config.BUPA_TARGET
assert config.BUPA_EXCLUDED not in frame.columns

train, test = splits.row_split(frame)
print(pandas.Series(splits.report(train, test)))
x_train, y_train = train[predictors].to_numpy(dtype=float), train[target].to_numpy(dtype=float)
x_test, y_test = test[predictors].to_numpy(dtype=float), test[target].to_numpy(dtype=float)

train_rows            255.0000
test_rows              86.0000
test_fraction           0.2522
rows_on_both_sides      0.0000
dtype: float64


## Training: least squares, a log-transformed fit, and a random forest

Three models on the same partition, so a weak result can be attributed to its
cause.

In [4]:
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor

ols = sm.OLS(y_train, sm.add_constant(x_train)).fit()
print("F = {:.2f} on {} and {} df, p = {:.2g}".format(
    ols.fvalue, int(ols.df_model), int(ols.df_resid), ols.f_pvalue))
intervals = ols.conf_int()
pandas.DataFrame({"estimate": ols.params, "ci_low": intervals[:, 0],
                  "ci_high": intervals[:, 1], "p": ols.pvalues},
                 index=["intercept"] + predictors).round(4)

F = 10.07 on 5 and 249 df, p = 8.6e-09


,estimate,ci_low,ci_high,p
intercept,-13.0257,-20.5301,-5.5212,0.0007
mcv,0.1670,0.0834,0.2506,0.0001
alkphos,0.0040,-0.0167,0.0248,0.7026
sgpt,-0.0105,-0.0390,0.0181,0.4715
sgot,0.0293,-0.0276,0.0863,0.3110
gammagt,0.0202,0.0095,0.0310,0.0003


In [5]:
log_ols = sm.OLS(numpy.log1p(y_train), sm.add_constant(x_train)).fit()
forest = RandomForestRegressor(n_estimators=500, min_samples_leaf=5,
                               random_state=config.SEED, n_jobs=-1).fit(x_train, y_train)

predictions = {
    "least squares": ols.predict(sm.add_constant(x_test)),
    "least squares on log(1 + drinks)": numpy.expm1(log_ols.predict(sm.add_constant(x_test))),
    "random forest": forest.predict(x_test),
}
pandas.DataFrame({name: evaluate.regression(y_test, predicted)
                  for name, predicted in predictions.items()}).loc[["r2", "rmse", "mae"]].round(3)

,least squares,least squares on log(1 + drinks),random forest
r2,0.216,0.153,0.235
rmse,3.099,3.221,3.061
mae,2.458,2.322,2.391


## Results

The five tests jointly relate to reported intake, F = 10.07 with
p = 8.6 × 10⁻⁹, and explain little of it. Mean corpuscular volume and
gamma-glutamyl transpeptidase are the two terms whose intervals exclude zero,
and they are the two assays used clinically to detect sustained drinking. The
log-transformed target scores lower and the forest slightly higher, so
neither the skew of the target nor the linear form is what limits the fit.

![Observed against predicted intake on the held-out subjects, and the distribution of held-out R² over twenty redrawn splits.](../figures/fig10_bupa_observed_predicted.png)

![The five coefficients with 95 percent confidence intervals.](../figures/fig11_bupa_coefficients.png)

### The same fit under repeated partitions

The pipeline redrew the split twenty times and refitted least squares on each.

In [6]:
repeats = pandas.read_csv(RESULTS / "m04_repeated_splits.csv")
print(repeats["r2"].agg(["mean", "std", "min", "max"]).round(3))
repeats.sort_values("r2", ascending=False).reset_index(drop=True).head(6)

mean    0.136
std     0.098
min    -0.157
max     0.253
Name: r2, dtype: float64


,seed,r2,rmse,test_rows
0,20251213,0.253491,2.719295,86
1,20251207,0.252063,3.405892,86
2,20251222,0.247402,2.705159,86
3,20251214,0.232805,2.901791,86
4,20251206,0.215867,3.099111,86
5,20251208,0.192754,3.055709,86


## Discussion

The held-out R² of 0.216 on the primary split is the fifth highest of the
twenty draws. The mean over draws is 0.136 with a standard deviation of 0.098,
and one draw is negative, meaning the panel predicted intake worse than the
training mean would have on those 86 subjects. A single partition of 341 rows
is one draw from a distribution whose width is comparable to its center, and
the number to carry away is the mean.

The residuals have a skew of 1.14, so the confidence intervals above are
narrower than the true sampling uncertainty. The target is a self-report with
16 distinct values and no stated reference period, every subject is male, and
the design is cross-sectional. The write-up sets each of these out with the
numbers behind it.